<a href="https://colab.research.google.com/github/thisishasan/speech_processing/blob/main/assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [112]:
!pip -q install opensmile

In [113]:
!pip -q install pyannote.audio

In [114]:
import torch

In [115]:
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

In [116]:
HUGGINGFACE_ACCESS_TOKEN = ''

In [117]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HUGGINGFACE_ACCESS_TOKEN)

In [118]:
AUDIO_PATH = "/content/voice_assignment_01.wav"

In [119]:
diarization = pipeline(AUDIO_PATH)

/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


In [120]:
segments = []
for turn, speaker in diarization.speaker_diarization:
    segments.append({
        "start": turn.start,
        "end": turn.end,
        "duration": turn.end - turn.start,
        "speaker": speaker
    })

In [121]:
import pandas as pd

In [122]:
df_segments = pd.DataFrame(segments)
df_segments

,start,end,duration,speaker
0,0.891594,5.127219,4.235625,SPEAKER_00
1,5.785344,7.860969,2.075625,SPEAKER_00
2,8.316594,11.286594,2.970000,SPEAKER_00
3,11.708469,12.805344,1.096875,SPEAKER_00
4,12.855969,13.733469,0.877500,SPEAKER_00


In [123]:
speaker_durations = df_segments.groupby("speaker")["duration"].sum()
speaker_durations

speaker
SPEAKER_00    11.255625
Name: duration, dtype: float64

In [124]:
target_speaker = speaker_durations.idxmax()
target_speaker

'SPEAKER_00'

In [125]:
speaker_rows = df_segments[df_segments["speaker"] == target_speaker]
speaker_rows

,start,end,duration,speaker
0,0.891594,5.127219,4.235625,SPEAKER_00
1,5.785344,7.860969,2.075625,SPEAKER_00
2,8.316594,11.286594,2.970000,SPEAKER_00
3,11.708469,12.805344,1.096875,SPEAKER_00
4,12.855969,13.733469,0.877500,SPEAKER_00


In [126]:
import soundfile as sf

In [127]:
audio, sr = sf.read(AUDIO_PATH)

In [128]:
chunks = []
for _, row in speaker_rows.iterrows():
    start_sample = int(row["start"] * sr)
    end_sample = int(row["end"] * sr)
    chunks.append(audio[start_sample:end_sample])
chunks

[array([-0.30923462, -0.26391602, -0.22930908, ..., -0.20526123,
        -0.09106445, -0.11508179], shape=(186791,)),
 array([ 0.19824219,  0.26144409,  0.27767944, ..., -0.03610229,
         0.11602783,  0.12564087], shape=(91535,)),
 array([0.05093384, 0.08685303, 0.07015991, ..., 0.37567139, 0.35443115,
        0.34118652], shape=(130977,)),
 array([-0.04827881, -0.09591675, -0.12686157, ..., -0.51867676,
        -0.51965332, -0.53787231], shape=(48372,)),
 array([0.02520752, 0.03997803, 0.02386475, ..., 0.14172363, 0.04959106,
        0.02151489], shape=(38697,))]

In [129]:
import numpy as np

In [130]:
speaker_audio = np.concatenate(chunks)
speaker_audio

array([-0.30923462, -0.26391602, -0.22930908, ...,  0.14172363,
        0.04959106,  0.02151489], shape=(496372,))

In [131]:
SPEAKER_FILE = "/content/speaker_segment.wav"

In [132]:
sf.write(SPEAKER_FILE, speaker_audio, sr)

In [133]:
import opensmile

In [134]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.LowLevelDescriptors
)

In [135]:
mfcc_df = smile.process_file(SPEAKER_FILE)
mfcc_df

Loudness_sma3  \
file                         start                  end                                        
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000          2.599372   
                             0 days 00:00:00.010000 0 days 00:00:00.030000          3.011721   
                             0 days 00:00:00.020000 0 days 00:00:00.040000          3.324730   
                             0 days 00:00:00.030000 0 days 00:00:00.050000          3.483802   
                             0 days 00:00:00.040000 0 days 00:00:00.060000          3.852622   
...                                                                                      ...   
                             0 days 00:00:11.160000 0 days 00:00:11.180000          3.075482   
                             0 days 00:00:11.170000 0 days 00:00:11.190000          2.888193   
                             0 days 00:00:11.180000 0 days 00:00:11.200000          2.840340   
                             0 days 00:00:11.190000 0 days 00:00:11.210000          2.784528   
                             0 days 00:00:11.200000 0 days 00:00:11.255600907       2.735713   

                                                                               alphaRatio_sma3  \
file                         start                  end                                          
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000          -14.248620   
                             0 days 00:00:00.010000 0 days 00:00:00.030000          -11.730476   
                             0 days 00:00:00.020000 0 days 00:00:00.040000           -9.016803   
                             0 days 00:00:00.030000 0 days 00:00:00.050000          -10.726100   
                             0 days 00:00:00.040000 0 days 00:00:00.060000           -9.452489   
...                                                                                        ...   
                             0 days 00:00:11.160000 0 days 00:00:11.180000          -14.430039   
                             0 days 00:00:11.170000 0 days 00:00:11.190000          -15.219157   
                             0 days 00:00:11.180000 0 days 00:00:11.200000          -15.234216   
                             0 days 00:00:11.190000 0 days 00:00:11.210000          -15.452985   
                             0 days 00:00:11.200000 0 days 00:00:11.255600907       -15.688198   

                                                                               hammarbergIndex_sma3  \
file                         start                  end                                               
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000                26.292269   
                             0 days 00:00:00.010000 0 days 00:00:00.030000                22.951645   
                             0 days 00:00:00.020000 0 days 00:00:00.040000                17.174456   
                             0 days 00:00:00.030000 0 days 00:00:00.050000                16.875282   
                             0 days 00:00:00.040000 0 days 00:00:00.060000                13.728139   
...                                                                                             ...   
                             0 days 00:00:11.160000 0 days 00:00:11.180000                25.422201   
                             0 days 00:00:11.170000 0 days 00:00:11.190000                26.843176   
                             0 days 00:00:11.180000 0 days 00:00:11.200000                27.811241   
                             0 days 00:00:11.190000 0 days 00:00:11.210000                27.026337   
                             0 days 00:00:11.200000 0 days 00:00:11.255600907             26.331686   

                                                                               slope0-500_sma3  \
file                         start                  end                                          
/content/speaker_se